# MoleculeNet Interim Analysis

This notebook reproduces the current interim Phase 1 plots using a consistent model color scheme.

It expects `summary_partial.csv` in the same directory. Later, you can point `SUMMARY_PATH` at the final finished `summary.csv`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
SUMMARY_PATH = ROOT / "summary_partial.csv"
OUT_DIR = ROOT / "notebook_outputs"
OUT_DIR.mkdir(exist_ok=True)

PALETTE = {
    "interpremol_frozen": "#1b5e20",
    "chemeleon_frozen": "#1565c0",
    "chemeleon_finetune": "#42a5f5",
    "random_forest": "#ef6c00",
    "interpremol_finetune": "#c62828",
}
MODEL_ORDER = [
    "interpremol_frozen",
    "chemeleon_frozen",
    "chemeleon_finetune",
    "random_forest",
    "interpremol_finetune",
]

df = pd.read_csv(SUMMARY_PATH)
df["complete_group"] = df.groupby(["dataset", "split", "model"])["seed"].transform("nunique") == 3
complete = df[df["complete_group"]].copy()
agg = complete.groupby(["dataset", "split", "model", "metric_name"], as_index=False).agg(
    metric_mean=("metric_value", "mean"),
    metric_std=("metric_value", "std"),
    seeds=("seed", "nunique"),
)

print(f"rows: {len(df)}")
print(f"complete rows: {len(complete)}")
agg

In [ ]:
if not agg.empty:
    groups = agg[["dataset", "split", "metric_name"]].drop_duplicates().sort_values(["dataset", "split"])
    fig, axes = plt.subplots(len(groups), 1, figsize=(12, 3.8 * len(groups)), squeeze=False)
    for ax, (_, gs) in zip(axes.flatten(), groups.iterrows()):
        sub = (
            agg[(agg.dataset == gs.dataset) & (agg.split == gs.split)]
            .set_index("model")
            .reindex([m for m in MODEL_ORDER if m in agg.model.unique()])
            .dropna(subset=["metric_mean"])
            .reset_index()
        )
        colors = [PALETTE[m] for m in sub["model"]]
        ax.bar(sub["model"], sub["metric_mean"], yerr=sub["metric_std"].fillna(0.0), capsize=4, color=colors, edgecolor="black", linewidth=0.6)
        ax.set_title(f"{gs.dataset.upper()} | {gs.split} | {gs.metric_name}", fontsize=12, weight="bold")
        ax.set_ylabel(gs.metric_name)
        ax.grid(axis="y", linestyle=":", alpha=0.35)
        ax.set_axisbelow(True)
        ax.tick_params(axis="x", rotation=22)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "completed_group_bars_consistent.png", dpi=200, bbox_inches="tight")
    plt.show()

In [ ]:
if not agg.empty:
    pivot = agg.pivot_table(index=["dataset", "split"], columns="model", values="metric_mean")
    pivot = pivot.reindex(columns=[m for m in MODEL_ORDER if m in pivot.columns])
    fig, ax = plt.subplots(figsize=(10, max(3, len(pivot) * 0.7)))
    im = ax.imshow(pivot.values, aspect="auto", cmap="YlGnBu")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=25, ha="right")
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f"{d} | {s}" for d, s in pivot.index])
    ax.set_title("Interim mean metric by completed dataset/split group", fontsize=12, weight="bold")
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("mean metric")
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v = pivot.values[i, j]
            if pd.notna(v):
                ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=8, color="black")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "completed_group_heatmap.png", dpi=200, bbox_inches="tight")
    plt.show()

pivot